In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader€€
# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors


# Convert training data to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

#we will Convert testing data to tensors
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)



In [ ]:
# 2. Create TensorDataset objects


# we will Create training dataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

# Create testing dataset
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)



In [ ]:
# 3. Create DataLoaders


# Create DataLoader for training data
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True)

# We Create DataLoader for testing data
test_loader = DataLoader(
test_dataset, batch_size=32, shuffle=False)




In [ ]:
# 4. Print shape of one batch
# Get the first batch from the training DataLoader
X_batch, y_batch = next(iter(train_loader))

# Print shapes
print("X batch shape:", X_batch.shape)
print("y batch shape:", y_batch.shape)




In [ ]:
# 5. Display sample images


# Get one batch
X_batch, y_batch = next(iter(train_loader))

# Number of images to display
num_images = 5

plt.figure(figsize=(12, 3))

for i in range(num_images):
    plt.subplot(1, num_images, i + 1)

    # Convert tensor from (C, H, W) to (H, W, C)
    img = X_batch[i].permute(1, 2, 0).numpy()

    plt.imshow(img)
    plt.title(f"Age: {y_batch[i].item()}")
    plt.axis("off")

plt.show()

In [ ]:
# Task 1: Write your model class here:


class AgeRegressionModel(nn.Module):
    def __init__(self):
        super(AgeRegressionModel, self).__init__()

        # Flatten layer to convert image tensor to vector
        self.flatten = nn.Flatten()

        # 4 linear layers
        self.fc1 = nn.Linear(3 * 36 * 36, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 64)
        self.fc4 = nn.Linear(64, 1)  # Output layer (age) And we put 1

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc4(self.relu(self.fc3(x)))  # No activation for regression
        return x

        plt.tight_layout() # Adjust the spacing between subplots to prevent overlap

plt.show()
# Display the plotted images


In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
    # Set the model to training mode
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        # Flatten images and move data to the selected device
        X_batch = X_batch.view(X_batch.size(0), -1).to(device)
        y_batch = y_batch.to(device)

        # Forward pass
        outputs = model(X_batch).squeeze(1)  # shape: (batch_size)

        # Compute loss
        loss = criterion(outputs, y_batch)

        # Backward pass and optimization
        optimizer.zero_grad()   # Clear previous gradients
        loss.backward()         # Compute gradients
        optimizer.step()        # Update model parameters

        running_loss += loss.item()

    # Average loss over all batches
    avg_loss = running_loss / len(train_loader)

    return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
def validate_one_epoch(model, criterion, val_loader, device):
    # Set the model to evaluation mode
    model.eval()

    running_loss = 0.0

    # Disable gradient calculation for validation
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            # Flatten images and move data to the selected device
            X_batch = X_batch.view(X_batch.size(0), -1).to(device)
            y_batch = y_batch.to(device)

            # Forward pass
            outputs = model(X_batch).squeeze(1)  # shape: (batch_size)

            # Compute validation loss
            loss = criterion(outputs, y_batch)

            running_loss += loss.item()

    # Average validation loss over all batches
    avg_val_loss = running_loss / len(val_loader)

    return avg_val_loss

In [ ]:
# Task 4: Define device, model, loss, optimizer:

# Select device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the model and move it to the selected device
model = AgeRegressionModel().to(device)

# Define loss function (Mean Squared Error for regression)
criterion = nn.MSELoss()

# Define optimizer (Adam optimizer)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Task 5: Start training for 20 epochs:
num_epochs = 20

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # Train for one epoch
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

    # Validate for one epoch
    val_loss = validate_one_epoch(model, criterion, test_loader, device)

    # Store losses
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Print progress
    print(f"Epoch [{epoch+1}/{num_epochs}] - "
          f"Train Loss: {train_loss:.4f}, "
          f"Validation Loss: {val_loss:.4f}")

In [ ]:
# Task 1: Write your code here:


# Create range of epochs
epochs = range(1, len(train_losses) + 1)

# Plot losses
plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, label="Training Loss")
plt.plot(epochs, val_losses, label="Validation Loss")

# Add labels and title
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss over Epochs")
plt.legend()

# Adjust layout and show plot
plt.tight_layout()
plt.show()

In [ ]:
#The plot shows that both training and validation losses decrease over epochs,
#indicating that the model is learning,
#with minor validation fluctuations due to data variability.


In [ ]:
# Task 2 (Bonus): Write your code here:


# Set model to evaluation mode
model.eval()

# Get one batch from the test loader
X_batch, y_batch = next(iter(test_loader))

# Move data to device
X_batch = X_batch.to(device)
y_batch = y_batch.to(device)

# Predict ages
with torch.no_grad():
    outputs = model(X_batch.view(X_batch.size(0), -1)).squeeze(1)

# Number of images to display
num_images = 5

plt.figure(figsize=(12, 3))

for i in range(num_images):
    plt.subplot(1, num_images, i + 1)

    # Convert image from tensor (C, H, W) to (H, W, C)
    img = X_batch[i].cpu().permute(1, 2, 0).numpy()

    plt.imshow(img)
    plt.title(f"Actual: {y_batch[i].item():.0f}\nPred: {outputs[i].item():.0f}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Display sample test images with their actual and predicted ages to visually
# evaluate the regression model's performance.
#
# The model shows larger errors for babies, where it often predicts a higher
# age than the actual one. This is expected because very young faces have
# less distinctive facial features.
#
# For middle-aged faces, the predictions are generally closer to the true ages.
# Larger errors may still appear for very young or older faces due to the low
# image resolution and the simplicity of the regression model.
